In [34]:
import json

with open('../results/resultsTest_0305111436_1_llama3170b_llama3170b.json', 'r') as file:
    data = json.load(file)

In [35]:
gameInfo = data.copy()

codemasterConfig = gameInfo["modelDetailsCodemaster"]
guesserConfig = gameInfo["modelDetailsGuesser"]

gameSetup = gameInfo["gameSetup"]
gameSetup["gameSize"]["total"] = sum(gameSetup["gameSize"].values())
# del gameSetup["board"]

games = gameInfo["games"]
for game in games:
    for round in game["rounds"]:
        del round["clue"]
        for guess in round["guesses"]:
            print(guess, guess["score"])
            guess = guess["score"]


refinement = gameInfo["refinements"]

{'word': 'dragon', 'group': 'blue', 'score': 1} 1
{'word': 'face', 'group': 'blue', 'score': 1} 1
{'word': 'face', 'group': 'blue', 'score': 1} 1
{'word': 'pupil', 'group': 'blue', 'score': 1} 1


In [32]:
games[0]["rounds"]

[{'count': 1, 'guesses': [{'word': 'dragon', 'group': 'blue', 'score': 1}]}]

In [ ]:
import json
import pandas as pd

# 1. Load data
with open('../results/resultsTest_0305111436_0_llama3170b_llama3170b.json', 'r') as file:
    data = json.load(file)

# 2. Extract Metadata
benchmarkId = data.get("benchmark", {})
codemaster_config = data.get("modelDetailsCodemaster", {})
del codemaster_config["prompt"]
del codemaster_config["final_strategy"]
guesser_config = data.get("modelDetailsGuesser", {})
del guesser_config["prompt"]
del guesser_config["final_strategy"]
game_setup = data.get("gameSetup", {})

# Calculate total game size and remove the bulky board data as you originally noted
total_game_size = sum(game_setup.get("gameSize", {}).values())
if "board" in game_setup:
    del game_setup["board"]

# 3. Flatten metadata into a single base dictionary
# We use prefixes to keep column names clear and distinct
base_metadata = {
    "benchmarkId":benchmarkId,
    "setup_totalGameSize": total_game_size
    }

for key, value in codemaster_config.items():
    base_metadata[f"cm_{key}"] = value

for key, value in guesser_config.items():
    base_metadata[f"guesser_{key}"] = value

for key, value in game_setup.items():
    # Handle the nested 'gameSize' dictionary specifically to keep it flat
    if isinstance(value, dict):
        for sub_key, sub_value in value.items():
            base_metadata[f"setup_{key}_{sub_key}"] = sub_value
    else:
        base_metadata[f"setup_{key}"] = value

# 4. Accumulate rows
accumulated_data = []

for game_idx, game in enumerate(data.get("games", [])):
    for round_idx, round_data in enumerate(game.get("rounds", [])):
        for guess in round_data.get("guesses", []):
            
            # Start with the guess-specific data
            row = {
                "game_index": game_idx,
                "round_index": round_idx,
                # "guessed_word": guess.get("word", guess.get("guess", "N/A")),
                "score": guess.get("score")
            }
            
            # Merge the base metadata into this row
            row.update(base_metadata)
            accumulated_data.append(row)

# 5. Convert to DataFrame and save
df = pd.DataFrame(accumulated_data)
df.to_csv('accumulated_results_with_metadata.csv', index=False)

print(f"Successfully processed {len(df)} guesses.")
print("Columns included:", df.columns.tolist())

Successfully processed 6 guesses.
Columns included: ['game_index', 'round_index', 'guessed_word', 'score', 'benchmarkId', 'setup_totalGameSize', 'cm_name', 'cm_type', 'guesser_name', 'guesser_type', 'setup_gameSize_blue', 'setup_gameSize_red', 'setup_gameSize_assassin', 'setup_languageConfig_German', 'setup_languageConfig_English']
